# Replace `still_needs_review` Species Folders

For each species folder inside `batch_XX_manual_review/still_needs_review/`:
1. Find the matching JSON file in `needs_text_regeneration/` (matched by species name)
2. Delete all existing files inside the species folder
3. Copy the regeneration JSON into the now-empty species folder

**Source:** `fixed_batches_20_GSI_all_entities/batch_XX_manual_review/still_needs_review/<Species>/`  
**Replacement files from:** `needs_text_regeneration/<NNNN>_<Species>_<model>.json`

In [1]:
from pathlib import Path
import shutil

base_dir  = Path("../fixed_batches_20_GSI_all_entities")
regen_dir = Path("../needs_text_regeneration")

# Build a lookup: species_name -> regen JSON path
# Filename pattern: {id}_{Genus}_{species}_{model}.json
regen_index = {}
for f in regen_dir.iterdir():
    if f.suffix != ".json":
        continue
    parts = f.stem.split("_")
    if len(parts) >= 3:
        species_name = f"{parts[1]}_{parts[2]}"
        regen_index[species_name] = f

print(f"Regeneration files indexed: {len(regen_index)}")
for sp, f in sorted(regen_index.items()):
    print(f"  {sp:45s} <- {f.name}")

Regeneration files indexed: 18
  Aphanius_dispar                               <- 0457_Aphanius_dispar_std.json
  Aythya_affinis                                <- 0679_Aythya_affinis_std.json
  Carcharhinus_albimarginatus                   <- 1080_Carcharhinus_albimarginatus_stf.json
  Coryphaena_equiselis                          <- 1687_Coryphaena_equiselis_abj.json
  Cyprogenia_stegaria                           <- 1902_Cyprogenia_stegaria_abj.json
  Geosciurus_princeps                           <- 2739_Geosciurus_princeps_stx.json
  Gobiesox_rhessodon                            <- 2832_Gobiesox_rhessodon_std.json
  Habrophlebia_lauta                            <- 2919_Habrophlebia_lauta_hep.json
  Hynobius_kimurae                              <- 3162_Hynobius_kimurae_std.json
  Larus_californicus                            <- 3421_Larus_californicus_std.json
  Macrobrachium_panamense                       <- 3838_Macrobrachium_panamense_abj.json
  Platycephalus_conatus             

In [2]:
replaced     = []
not_matched  = []

for batch_manual in sorted(base_dir.iterdir()):
    if not batch_manual.is_dir() or "manual_review" not in batch_manual.name:
        continue
    snr_dir = batch_manual / "still_needs_review"
    if not snr_dir.exists():
        continue

    for species_dir in sorted(snr_dir.iterdir()):
        if not species_dir.is_dir():
            continue
        species_name = species_dir.name

        if species_name not in regen_index:
            not_matched.append(str(species_dir))
            continue

        regen_file = regen_index[species_name]

        # Remove all existing files in the species folder
        for old_file in species_dir.iterdir():
            if old_file.is_file():
                old_file.unlink()

        # Copy the regeneration JSON into the now-empty folder
        dest = species_dir / regen_file.name
        shutil.copy2(regen_file, dest)
        replaced.append((batch_manual.name, species_name, regen_file.name))

print(f"Species folders replaced: {len(replaced)}")
print()
for batch, species, fname in replaced:
    print(f"  [{batch}/still_needs_review/{species}]  <-  {fname}")

if not_matched:
    print(f"\nWarning — {len(not_matched)} species folder(s) had no matching regen file:")
    for p in not_matched:
        print(f"  {p}")

Species folders replaced: 18

  [batch_02_manual_review/still_needs_review/Aphanius_dispar]  <-  0457_Aphanius_dispar_std.json
  [batch_02_manual_review/still_needs_review/Aythya_affinis]  <-  0679_Aythya_affinis_std.json
  [batch_03_manual_review/still_needs_review/Carcharhinus_albimarginatus]  <-  1080_Carcharhinus_albimarginatus_stf.json
  [batch_05_manual_review/still_needs_review/Coryphaena_equiselis]  <-  1687_Coryphaena_equiselis_abj.json
  [batch_06_manual_review/still_needs_review/Cyprogenia_stegaria]  <-  1902_Cyprogenia_stegaria_abj.json
  [batch_08_manual_review/still_needs_review/Geosciurus_princeps]  <-  2739_Geosciurus_princeps_stx.json
  [batch_08_manual_review/still_needs_review/Gobiesox_rhessodon]  <-  2832_Gobiesox_rhessodon_std.json
  [batch_08_manual_review/still_needs_review/Habrophlebia_lauta]  <-  2919_Habrophlebia_lauta_hep.json
  [batch_09_manual_review/still_needs_review/Hynobius_kimurae]  <-  3162_Hynobius_kimurae_std.json
  [batch_10_manual_review/still_nee

In [3]:
# Sanity check — list contents of each replaced species folder
print("Contents after replacement:\n")
for batch, species, _ in replaced:
    folder = base_dir / batch / "still_needs_review" / species
    files  = sorted(f.name for f in folder.iterdir() if f.is_file())
    print(f"  {batch}/still_needs_review/{species}/")
    for fname in files:
        print(f"    {fname}")

Contents after replacement:

  batch_02_manual_review/still_needs_review/Aphanius_dispar/
    0457_Aphanius_dispar_std.json
  batch_02_manual_review/still_needs_review/Aythya_affinis/
    0679_Aythya_affinis_std.json
  batch_03_manual_review/still_needs_review/Carcharhinus_albimarginatus/
    1080_Carcharhinus_albimarginatus_stf.json
  batch_05_manual_review/still_needs_review/Coryphaena_equiselis/
    1687_Coryphaena_equiselis_abj.json
  batch_06_manual_review/still_needs_review/Cyprogenia_stegaria/
    1902_Cyprogenia_stegaria_abj.json
  batch_08_manual_review/still_needs_review/Geosciurus_princeps/
    2739_Geosciurus_princeps_stx.json
  batch_08_manual_review/still_needs_review/Gobiesox_rhessodon/
    2832_Gobiesox_rhessodon_std.json
  batch_08_manual_review/still_needs_review/Habrophlebia_lauta/
    2919_Habrophlebia_lauta_hep.json
  batch_09_manual_review/still_needs_review/Hynobius_kimurae/
    3162_Hynobius_kimurae_std.json
  batch_10_manual_review/still_needs_review/Larus_cali

In [4]:

# Copy the matching Species_name.json from species_data into each still_needs_review species folder
species_data_dir = Path("../../data-generation/species_data")

copied_sd   = []
missing_sd  = []

for batch_manual in sorted(base_dir.iterdir()):
    if not batch_manual.is_dir() or "manual_review" not in batch_manual.name:
        continue
    snr_dir = batch_manual / "still_needs_review"
    if not snr_dir.exists():
        continue

    for species_dir in sorted(snr_dir.iterdir()):
        if not species_dir.is_dir():
            continue
        species_name = species_dir.name
        src = species_data_dir / f"{species_name}.json"

        if not src.exists():
            missing_sd.append(species_name)
            continue

        dest = species_dir / src.name
        if dest.exists():
            copied_sd.append((species_name, "(already exists, skipped)"))
            continue

        shutil.copy2(src, dest)
        copied_sd.append((species_name, src.name))

print(f"species_data files copied: {len([x for x in copied_sd if 'skipped' not in x[1]])}")
print()
for species, fname in copied_sd:
    print(f"  {species:45s}  ->  {fname}")

if missing_sd:
    print(f"\nWarning — {len(missing_sd)} species had no match in species_data/:")
    for s in missing_sd:
        print(f"  {s}")


species_data files copied: 18

  Aphanius_dispar                                ->  Aphanius_dispar.json
  Aythya_affinis                                 ->  Aythya_affinis.json
  Carcharhinus_albimarginatus                    ->  Carcharhinus_albimarginatus.json
  Coryphaena_equiselis                           ->  Coryphaena_equiselis.json
  Cyprogenia_stegaria                            ->  Cyprogenia_stegaria.json
  Geosciurus_princeps                            ->  Geosciurus_princeps.json
  Gobiesox_rhessodon                             ->  Gobiesox_rhessodon.json
  Habrophlebia_lauta                             ->  Habrophlebia_lauta.json
  Hynobius_kimurae                               ->  Hynobius_kimurae.json
  Larus_californicus                             ->  Larus_californicus.json
  Macrobrachium_panamense                        ->  Macrobrachium_panamense.json
  Platycephalus_conatus                          ->  Platycephalus_conatus.json
  Proteles_cristatus             